In [7]:
!pip install utils

In [8]:
!pip install openai

In [9]:
!pip install llama_index

In [10]:
!pip install llama-index.core

In [11]:
!pip install llama-index-llms-openai

In [12]:
#!pip install llama-index-embeddings-huggingface

In [13]:
#!pip install llama_index.embeddings.huggingface

In [14]:
#!pip install llama_index.llms.huggingface_api

In [15]:
#!pip install llama_index.llms.ollama

In [16]:
#!pip install llama_index.llms.anthropic

In [17]:
#!pip install torch

In [18]:
#!pip install accelerate

In [19]:
#!pip install git+https://github.com/huggingface/accelerate

**BUILDING A MULTILINGUAL SPEECH RECOGNITION MODEL FOR RAG WITHOUT TRAINING.**

In [1]:
import os
import openai
import utils

In [2]:
from llama_index.core import SimpleDirectoryReader

Starting the process by installing and importing all the libraries necessary for the fucntioning of the process.

In [3]:
documents = SimpleDirectoryReader(
    input_files=["/content/Spotify Info.pdf"]
).load_data()

The file imported here has the information about Spotify and how Spotify is using AI. It's a PDF document used to provide information to the system so that we get answers for the queries we are asking in any language .

In [4]:
print(type(documents), "\n")
print(len(documents), "\n")
print(type(documents[0]))
print(documents[0])

<class 'list'> 

5 

<class 'llama_index.core.schema.Document'>
Doc ID: c5d92bd7-1147-4b40-9951-85f64971fa30
Text: C.Jahnavi.Lalasa   20MIA1114  MGT3007 – RETAIL ANALYTICS
DIGITAL ASSIGNMENT – 1  AI TRENDS PRACTICED BY SPOTIFY    Company
Taken: SPOTIFY   Spotify is the world's leading audio streaming
platform. As of June 2024, Spotify  remains the undisputed champion in
the audio streaming arena, boasting a staggering  user base
approaching 700 million stro...


This gives us the information of our document, its length and the contents its holding.

In [5]:
from llama_index.core import  Document

document = Document(text="\n\n".join([doc.text for doc in documents]))

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader,Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.llms.ollama import Ollama
from llama_index.llms.anthropic import Anthropic
HF_TOKEN='use your hugging face token' 



Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")
Settings.llm =HuggingFaceInferenceAPI(model_name="HuggingFaceH4/zephyr-7b-alpha", token=HF_TOKEN)

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")
llm =HuggingFaceInferenceAPI(model_name="HuggingFaceH4/zephyr-7b-alpha", token=HF_TOKEN)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when poss

RAG is the process we use to obtain optimized output of a large language model.Meaning it gives us meaningful output for the queries we are asking with whatever pre-trained knowledge it has, which serves as a benefit to the already highly capable LLMs.

RAG has three major steps of Ingestion , Retrivel and Synthesis.
Here we can see the process of Ingestion .The first step in ingestion is to chunck the data and perform the embedding.Here we are using the "llama_Index" to input the data and the features of llama_index perform the NLP tasks,help text analysis,information retreival and divide them into understandable chunks by the system.

HF_Toekn is used to connect with Hugging Face API token which is used for for authentication and usage of Hugging Face's services.The 'settings.embed_model sets up the embedding model to use for vector embeddings.

In [7]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from accelerate import Accelerator
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model_id = "openai/whisper-large-v3"
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)
processor = AutoProcessor.from_pretrained(model_id)
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    max_new_tokens=128,
    chunk_length_s=30,
    batch_size=16,
    return_timestamps=True,
    torch_dtype=torch_dtype,
    device=device,
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


The next process is we are setting up the complete pipeline for automatic speech reconition using a pretrained model "openai/Whisper-large-v3" from hugging face.It manages the computation , tells us the number of tokens the data was divided into ,the chucnk length and configures various parameters for ASR inference.

In [8]:
result = pipe("/content/telugu.question.aac",generate_kwargs={"task": "translate"})
print(result["text"])

 What is Spotify?


An audio file in the language telugu is uploaded here which is translated and shown.

In [9]:
index = VectorStoreIndex.from_documents(
    documents,
)

Indexing is done for the document that was provided.

In [10]:
query_engine = index.as_query_engine()

Earlier we provided an audio file which was translated and this is where we are using it as a query statement to get our solution and check the working of RAG system.

This the step where RETRIVEL is done.

In [11]:
response = query_engine.query(
    "What is spotify?"
)
print(str(response))


Spotify is the world's leading audio streaming platform. As of June 2024, Spotify 
remains the undisputed champion in the audio streaming arena, boasting a staggering 
user base approaching 700 million strong. Their ever-expanding library explodes with 
over 100 million songs, catering to every musical taste. But music isn't all they offer - 
a thriving podcast and audiobook scene makes Spotify a one-stop shop for all your 
audio needs. AI plays a crucial role, constantly refining music recommendations with 
features like Discover Weekly acting as your personal DJ. The freemium model might 
undergo adjustments, with targeted advertising or limited features potentially 
impacting the free tier. However, Spotify stays ahead of the curve by securing 
exclusive content deals with artists and podcasters, ensuring a truly unique and 
personalized audio experience.


The resposnse is received.

In [12]:
eval_questions = []


In [13]:
new_question = pipe("/content/telugu.question2.aac",generate_kwargs={"task": "translate"})
eval_questions.append(new_question)

Now we know that the RAG system is working to check its efficiency I provided the system with another audio file which would be used for querying .

In [14]:
print(eval_questions)

[{'text': ' Do we use AI in Spotify? How?', 'chunks': [{'timestamp': (0.0, 3.5), 'text': ' Do we use AI in Spotify? How?'}]}]


the second audio file which was in telugu language is shown to be translated in English here.

In [15]:
response2 = query_engine.query(
    "eval_questions"
)
print(str(response2))



1. What is the role of AI in Spotify's recommendation engine?

Answer: AI plays a significant role in Spotify's recommendation engine by analyzing user data, listening habits, and preferences to provide personalized recommendations. The AI algorithms categorize music into different moods and activities, such as "chill," "workout," or "study," making it easier for users to find the right music for any situation. The AI also adapts playlists in real-time to keep them fresh and relevant, ensuring that users always have something new to listen to.

2. How does Spotify's AI analyze raw audio tracks?

Answer: Spotify uses convolutional neural networks to analyze raw audio tracks. This ensures that even new or less popular tracks get recommended based on their audio characteristics rather than their online presence.

3. What are some examples of personalized recommendations provided by Spotify's AI?

Answer: Spotify's AI provides personalized recommendations through various features, includ

Here we can see that the second query we provided has given results.
These results here shows us the final step of RAG SYSNTHESIS.

Conclusion:
By doing this assignment we built a multilingual speech recognition model for RAG without training it. For performing this task we used multilingual Whisper for speech recognition which didnt require any traning and for building RAG we used llama_index which provided diffierent librabries that helped for NLP processes, text analysis , hugging face services whihch was used to connect the LLM model. At the end when an audio file in the language Telugu was provided we could succesfully translate it to english and get solutionsEnglish for the queries making the model successful.